# CNN-GAT — Julian's 1D CNN + Graph Attention Network

Personal experiment — separate from the group submission.

**Strategy:** Use Julian's best 1D CNN (with attention) as temporal feature extractor per sensor, then add GAT layers to model spatial sensor relationships.

**Architecture:**
1. Julian's 1D CNN + attention (temporal feature extraction per sensor)
2. Spectral band power features (delta, theta, alpha, beta, gamma)
3. GAT layers — learns spatial relations between 248 sensors
4. Global mean pooling over nodes
5. Dense(4) classification

## 1. Data Loading

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

CLASSES = ["rest", "motor", "math", "memory"]

def load_dataset(name):
    return np.load(f"Preprocessed data/{name}.npy")

X_train     = load_dataset("X_intra_train_w")  # (512, 248, 500)
X_test      = load_dataset("X_intra_test_w")   # (64, 248, 500)
y_train     = load_dataset("y_intra_train_w")  # (512,)
y_test      = load_dataset("y_intra_test_w")   # (64,)
y_test_orig = load_dataset("y_intra_test")     # (8,) original recordings

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)

## 2. Spectral Band Power Features

In [ ]:
from scipy.signal import butter, sosfiltfilt

bands = {'delta': (1,4), 'theta': (4,8), 'alpha': (8,13), 'beta': (13,30), 'gamma': (30,50)}
fs = 254.0

def extract_band_power(X, fs, bands):
    n_samples, n_sensors, n_times = X.shape
    bp = np.zeros((n_samples, n_sensors, len(bands)), dtype='float32')
    for b_idx, (_, (f_low, f_high)) in enumerate(bands.items()):
        sos = butter(4, [f_low, f_high], btype='band', fs=fs, output='sos')
        for s in range(n_samples):
            for ch in range(n_sensors):
                filtered = sosfiltfilt(sos, X[s, ch])
                bp[s, ch, b_idx] = np.var(filtered)
    return bp

print("Extracting band power features (this takes a few minutes)...")
bp_train = extract_band_power(X_train, fs, bands)  # (512, 248, 5)
bp_test  = extract_band_power(X_test,  fs, bands)  # (64, 248, 5)
print("Train band power:", bp_train.shape)
print("Test band power: ", bp_test.shape)

## 3. Adjacency Matrix (Distance-based)

In [ ]:
import mne
from scipy.spatial.distance import cdist

layout      = mne.channels.read_layout('magnesWH3600')
pos         = layout.pos[:, :2]
dist_matrix = cdist(pos, pos, metric='euclidean')
sigma       = dist_matrix.std()
A_distance  = np.exp(-dist_matrix**2 / (2 * sigma**2))
np.fill_diagonal(A_distance, 0)

A_hat      = A_distance + np.eye(248)
degree     = A_hat.sum(axis=1)
D_inv_sqrt = np.diag(1.0 / np.sqrt(degree))
A_norm     = D_inv_sqrt @ A_hat @ D_inv_sqrt
A_t        = torch.tensor((A_norm > 0.01).astype(float) * A_norm, dtype=torch.float32)

print("Adjacency matrix shape:", A_t.shape)

## 4. Model Architecture

### Temporal feature extractor
Julian's 1D CNN with attention (`build_1d_cnn` with `attention=True` from `cnn_tests.ipynb`), re-implemented in PyTorch. Applied independently per sensor.

### GAT layers
Graph Attention Network (Veličković et al. 2018, Lecture 8 slides 423-425).

In [ ]:
class TemporalAttention1D(nn.Module):
    """Julian's temporal attention — weights each timestep."""
    def __init__(self, in_features):
        super().__init__()
        self.attn = nn.Linear(in_features, 1)

    def forward(self, x):
        # x: (B, T, C)
        weights = torch.tanh(self.attn(x))          # (B, T, 1)
        weights = torch.softmax(weights, dim=1)      # (B, T, 1)
        return x * weights                           # (B, T, C)


class SqueezeExcite1D(nn.Module):
    """Julian's squeeze-excite channel attention."""
    def __init__(self, channels, ratio=8):
        super().__init__()
        self.fc1 = nn.Linear(channels, max(1, channels // ratio))
        self.fc2 = nn.Linear(max(1, channels // ratio), channels)

    def forward(self, x):
        # x: (B, T, C)
        se = x.mean(dim=1)                           # (B, C)
        se = F.relu(self.fc1(se))                    # (B, C//ratio)
        se = torch.sigmoid(self.fc2(se))             # (B, C)
        return x * se.unsqueeze(1)                   # (B, T, C)


class JulianCNNBlock(nn.Module):
    """One Conv1D block from Julian's build_1d_cnn with attention=True."""
    def __init__(self, in_ch, out_ch, kernel_size, dropout=0.3):
        super().__init__()
        self.conv    = nn.Conv1d(in_ch, out_ch, kernel_size, padding=kernel_size//2)
        self.bn      = nn.BatchNorm1d(out_ch)
        self.pool    = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(dropout)
        self.t_attn  = TemporalAttention1D(out_ch)
        self.se      = SqueezeExcite1D(out_ch)

    def forward(self, x):
        # x: (B, C, T)
        x = F.relu(self.bn(self.conv(x)))            # (B, out_ch, T)
        x = self.pool(x)                             # (B, out_ch, T/2)
        x = self.dropout(x)
        x = x.transpose(1, 2)                        # (B, T/2, out_ch)
        x = self.t_attn(x)
        x = self.se(x)
        x = x.transpose(1, 2)                        # (B, out_ch, T/2)
        return x


class JulianTemporalExtractor(nn.Module):
    """
    Julian's 1D CNN with attention, re-implemented in PyTorch.
    Applied per sensor independently.
    Input:  (B*N, 1, T)
    Output: (B*N, 64) — 64 temporal features per sensor
    """
    def __init__(self, dropout=0.3):
        super().__init__()
        self.block1 = JulianCNNBlock(1,  16, 7, dropout)
        self.block2 = JulianCNNBlock(16, 32, 5, dropout)
        self.block3 = JulianCNNBlock(32, 64, 3, dropout)
        self.gap     = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        x = self.block1(x)              # (B*N, 16, T/2)
        x = self.block2(x)              # (B*N, 32, T/4)
        x = self.block3(x)              # (B*N, 64, T/8)
        return self.gap(x).squeeze(-1)  # (B*N, 64)


class GATLayer(nn.Module):
    """
    GAT layer — Veličković et al. 2018 (Lecture 8, slides 423-425)
    α_ij = softmax( LeakyReLU( a^T [Wh_i || Wh_j] ) )
    h'_i = ELU( Σ_j α_ij W h_j )
    """
    def __init__(self, in_features, out_features, dropout=0.3):
        super().__init__()
        self.W          = nn.Linear(in_features, out_features, bias=False)
        self.a_src      = nn.Linear(out_features, 1, bias=False)
        self.a_dst      = nn.Linear(out_features, 1, bias=False)
        self.leaky_relu = nn.LeakyReLU(0.2)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, H, A):
        B, N, _ = H.shape
        Wh    = self.W(H)
        e_src = self.a_src(Wh)
        e_dst = self.a_dst(Wh)
        e     = self.leaky_relu(e_src + e_dst.transpose(1, 2))
        mask  = (A == 0).unsqueeze(0).expand(B, -1, -1)
        e     = e.masked_fill(mask, float('-inf'))
        alpha = self.dropout(F.softmax(e, dim=-1))
        return F.elu(torch.bmm(alpha, Wh))


class CNN_GAT(nn.Module):
    def __init__(self, n_sensors=248, n_bands=5,
                 gat_hidden=32, n_classes=4, dropout=0.3):
        super().__init__()

        # Julian's temporal extractor (output: 64 features per sensor)
        self.extractor   = JulianTemporalExtractor(dropout=dropout)
        self.bn_temporal = nn.BatchNorm1d(64)

        # GAT layers — input = 64 temporal + 5 band power
        self.gat1 = GATLayer(64 + n_bands, gat_hidden, dropout)
        self.bn1  = nn.LayerNorm(gat_hidden)

        self.gat2 = GATLayer(gat_hidden, gat_hidden, dropout)
        self.bn2  = nn.LayerNorm(gat_hidden)

        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(gat_hidden, n_classes)

    def forward(self, X, A, BP):
        B, N, T = X.shape

        # Temporal extraction per sensor
        X = X.reshape(B * N, 1, T)
        X = self.extractor(X)                              # (B*N, 64)
        X = self.bn_temporal(X)
        X = X.reshape(B, N, -1)                            # (B, N, 64)

        # Concatenate band power
        X = torch.cat([X, BP], dim=-1)                     # (B, N, 69)

        # GAT layers
        X = self.bn1(self.gat1(X, A))                      # (B, N, gat_hidden)
        X = self.bn2(self.gat2(X, A))                      # (B, N, gat_hidden)

        # Global mean pooling + classify
        X = self.dropout(X.mean(dim=1))                    # (B, gat_hidden)
        return self.classifier(X)

## 5. Training

- Mini-batch (batch_size=32) with shuffling
- Early stopping on test loss (patience=20)
- Majority vote at recording level (8 recordings, 8 windows each)

In [ ]:
X_train_t  = torch.tensor(X_train,  dtype=torch.float32)
X_test_t   = torch.tensor(X_test,   dtype=torch.float32)
y_train_t  = torch.tensor(y_train,  dtype=torch.long)
y_test_t   = torch.tensor(y_test,   dtype=torch.long)
BP_train_t = torch.tensor(bp_train, dtype=torch.float32)
BP_test_t  = torch.tensor(bp_test,  dtype=torch.float32)

In [ ]:
dataset = TensorDataset(X_train_t, BP_train_t, y_train_t)
loader  = DataLoader(dataset, batch_size=32, shuffle=True)

model     = CNN_GAT()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

best_test_loss    = float('inf')
best_epoch        = 0
best_model_state  = None
patience          = 20
epochs_no_improve = 0

for epoch in range(150):
    model.train()
    for X_batch, BP_batch, y_batch in loader:
        optimizer.zero_grad()
        out  = model(X_batch, A_t, BP_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        out_train  = model(X_train_t, A_t, BP_train_t)
        loss_train = criterion(out_train, y_train_t).item()
        acc_train  = (out_train.argmax(1) == y_train_t).float().mean().item()

        out_test   = model(X_test_t, A_t, BP_test_t)
        loss_test  = criterion(out_test, y_test_t).item()
        acc_test   = (out_test.argmax(1) == y_test_t).float().mean().item()

    if loss_test < best_test_loss:
        best_test_loss    = loss_test
        best_epoch        = epoch + 1
        best_model_state  = {k: v.clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Train loss: {loss_train:.4f}  acc: {acc_train:.2%} | Test loss: {loss_test:.4f}  acc: {acc_test:.2%}")

    if epochs_no_improve >= patience:
        print(f"\nEarly stopping at epoch {epoch+1} (best epoch: {best_epoch})")
        break

model.load_state_dict(best_model_state)
print(f"\nBest model restored from epoch {best_epoch} (test loss: {best_test_loss:.4f})")

## 6. Evaluation — Majority Vote + Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

n_recordings    = 8
windows_per_rec = len(X_test) // n_recordings

model.eval()
with torch.no_grad():
    probs = torch.softmax(model(X_test_t, A_t, BP_test_t), dim=1).numpy()

mv_preds = []
for i in range(n_recordings):
    avg_prob = probs[i * windows_per_rec:(i + 1) * windows_per_rec].mean(axis=0)
    mv_preds.append(np.argmax(avg_prob))

mv_preds = np.array(mv_preds)
mv_acc   = np.mean(mv_preds == y_test_orig)
print(f"Majority vote accuracy (recording-level): {mv_acc:.2%}")

cm = confusion_matrix(y_test_orig, mv_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion matrix — majority vote ({mv_acc:.0%} accuracy)')
plt.tight_layout()
plt.show()

print(classification_report(y_test_orig, mv_preds, target_names=CLASSES, zero_division=0))